## Import Dependencies

In [ ]:
!git clone -b model https://github.com/tt219493/ESM-Secondary-Prediction.git
%cd /content/ESM-Secondary-Prediction
!pip install -r requirements.txt

In [15]:
import polars as pl
from transformers import AutoTokenizer
from preprocess import tokenize_benchmark, remove_overlapping_seq


## Import Data

In [3]:
token_train_path = "/content/tokenized_train.parquet"
token_test_path = "/content/tokenized_test.parquet"

In [4]:
token_train_file = pl.scan_parquet(token_train_path)
token_test_file = pl.scan_parquet(token_test_path)

In [5]:
bm_train_path = '/content/Train_HHblits.csv'
casp12_path = '/content/CASP12_HHblits.csv'
cb513_path = '/content/CB513_HHblits.csv'
new364_path = '/content/NEW364.csv'
ts115_path = '/content/TS115_HHblits.csv'

bm_train_file = pl.scan_csv(bm_train_path)
casp12_file = pl.scan_csv(casp12_path)
cb513_file = pl.scan_csv(cb513_path)
new364_file = pl.scan_csv(new364_path).rename({'dssp8': ' dssp8'})
ts115_file = pl.scan_csv(ts115_path)

## Process Data

### Pre-Tokenized Data

In [6]:
token_train_c9_file = (token_train_file
                       .with_columns(
                           secondary_structure=pl.when((pl.col('secondary_structure') == ' ') |
                                                       (pl.col('secondary_structure') == '.'))
                                                  .then(pl.lit('C'))
                                                  .otherwise(pl.col('secondary_structure')))
                       .with_columns(
                           label=pl.when(pl.col('label') == 1)
                                   .then(pl.lit(0))
                                   .otherwise(pl.col('label')))
                       )


In [7]:
token_test_c9_file = (token_test_file
                       .with_columns(
                           secondary_structure=pl.when((pl.col('secondary_structure') == ' ') |
                                                       (pl.col('secondary_structure') == '.'))
                                                  .then(pl.lit('C'))
                                                  .otherwise(pl.col('secondary_structure')))
                       .with_columns(
                           label=pl.when(pl.col('label') == 1)
                                   .then(pl.lit(0))
                                   .otherwise(pl.col('label')))
                       )

In [8]:
mapping = dict(
(token_test_c9_file
 .select(['secondary_structure', 'label'])
 .unique(['secondary_structure', 'label'])
 .sort(['label'])
 ).collect().iter_rows()
 )
print(mapping)

{'C': 0, 'B': 2, 'E': 3, 'G': 4, 'H': 5, 'I': 6, 'P': 7, 'S': 8, 'T': 9}


In [9]:
def aggregate_df(df):
  return (df.sort(['id', 'asym_id', 'index'])
                                  .group_by(['id', 'asym_id', 'sequence'])
                                  .agg(['secondary_structure', 'index', 'label', 'input_ids', 'attention_mask']))


In [10]:
token_train_df = aggregate_df(token_train_file)
token_test_df = aggregate_df(token_test_file)
token_train_c9_df = aggregate_df(token_train_c9_file)
token_test_c9_df = aggregate_df(token_test_c9_file)

### Benchmark Data

In [11]:
bm_train_df = tokenize_benchmark(bm_train_file, mapping)
casp12_df = tokenize_benchmark(casp12_file, mapping)
cb513_df = tokenize_benchmark(cb513_file, mapping)
new364_df = tokenize_benchmark(new364_file, mapping)
ts115_df = tokenize_benchmark(ts115_file, mapping)

Map:   0%|          | 0/10792 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/511 [00:00<?, ? examples/s]

Map:   0%|          | 0/364 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

## Remove Any Overlapping Sequences Between Two Datasets

In [20]:
test_df_list = [casp12_df, cb513_df, new364_df, ts115_df, token_test_df]

output_bm_train = remove_overlapping_seq(bm_train_df, test_df_list[:-1])

output_train = remove_overlapping_seq(token_train_df, test_df_list)
output_test = remove_overlapping_seq(token_test_df, [output_bm_train])

output_train_c9 = remove_overlapping_seq(token_train_c9_df, test_df_list)
output_test_c9 = remove_overlapping_seq(token_test_c9_df, [output_bm_train])


## Output Data

### Explode columns


In [21]:
output_train = (output_train
                .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_test = (output_test
                .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_train_c9 = (output_train_c9
                   .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_test_c9 = (output_test_c9
                  .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

In [22]:
output_bm_train = (output_bm_train
                   .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_casp12 = (casp12_df
                 .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_cb513 = (cb513_df
                .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_new364 = (new364_df
                 .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))

output_ts115 = (ts115_df
                .explode(['secondary_structure','index', 'label', 'input_ids', 'attention_mask']))


### Write to Parquet

In [23]:
output_train.collect().write_parquet('tokenized_train_unique.parquet')
output_test.collect().write_parquet('tokenized_test_unique.parquet')
output_train_c9.collect().write_parquet('tokenized_train_c9_unique.parquet')
output_test_c9.collect().write_parquet('tokenized_test_c9_unique.parquet')

In [24]:
output_bm_train.collect().write_parquet('Train_HHblits.parquet')
output_casp12.collect().write_parquet('CASP12_HHblits.parquet')
output_cb513.collect().write_parquet('CB513_HHblits.parquet')
output_new364.collect().write_parquet('NEW364.parquet')
output_ts115.collect().write_parquet('TS115_HHblits.parquet')